# Pandas for Data Analysis — Hands-on Guide

This notebook walks through the core Pandas operations every data analyst / AI engineer uses daily. Each section has a short explanation, the syntax, and a runnable example.

**Contents**
1. Importing Pandas & creating data
2. Selecting columns
3. Filtering rows
4. Changing data types
5. Creating new columns
6. Assigning values (`loc`, `iloc`, `at`)
7. Conditional selection with NumPy (`np.where`, `np.select`)
8. Sorting values
9. Removing duplicates
10. Imputing missing values
11. Pivoting (`pivot`, `pivot_table`, `melt`)
12. More essential functions (`rename`, `drop`, `concat`, `join`/`merge`, `replace`/`map`, `value_counts`)
13. Practice exercises

We use a small **loan portfolio** dataset so the examples feel realistic.

## 1. Importing Pandas

- **Pandas** provides two main structures:
  - `Series` → a single labelled column (1-D)
  - `DataFrame` → a table of rows and columns (2-D)
- **NumPy** is the numerical engine underneath Pandas; we use it for fast conditional logic and `NaN` (missing values).
- By convention: `import pandas as pd`, `import numpy as np`.

In [1]:
import pandas as pd
import numpy as np

print("Pandas version:", pd.__version__)
print("NumPy version :", np.__version__)

Pandas version: 3.0.2
NumPy version : 2.4.4


### 1.1 Creating a DataFrame

In real projects you would load data with `pd.read_csv('file.csv')` or `pd.read_excel('file.xlsx')`. Here we build one from a dictionary so the notebook runs anywhere.

Note: the data is intentionally *messy* — numbers stored as text, missing values, and a duplicate row — so we can clean it later.

In [2]:
data = {
    "loan_id":      ["L001", "L002", "L003", "L004", "L005", "L006", "L007", "L008", "L009", "L005"],
    "customer":     ["Rahim", "Karim", "Nusrat", "Tanvir", "Sadia", "Arif", "Mitu", "Jamal", "Farzana", "Sadia"],
    "branch":       ["Dhaka", "Chattogram", "Dhaka", "Sylhet", "Khulna", "Dhaka", "Sylhet", "Chattogram", "Khulna", "Khulna"],
    "sector":       ["Trading", "Manufacturing", "Service", "Trading", "Agriculture", "Service", "Trading", "Manufacturing", "Agriculture", "Agriculture"],
    "loan_amount":  ["500000", "1200000", "300000", "750000", "200000", "950000", "400000", "1500000", "250000", "200000"],
    "outstanding":  [420000, 1100000, np.nan, 600000, 180000, 900000, 150000, np.nan, 240000, 180000],
    "overdue_days": [0, 45, 10, 95, np.nan, 200, 0, 30, 400, np.nan],
    "disburse_date":["2024-01-15", "2023-06-20", "2024-03-10", "2022-11-05", "2024-05-01",
                     "2021-08-18", "2024-02-28", "2023-01-12", "2020-09-30", "2024-05-01"],
}

df = pd.DataFrame(data)
df

,loan_id,customer,branch,sector,loan_amount,outstanding,overdue_days,disburse_date
0,L001,Rahim,Dhaka,Trading,500000,420000.0,0.0,2024-01-15
1,L002,Karim,Chattogram,Manufacturing,1200000,1100000.0,45.0,2023-06-20
2,L003,Nusrat,Dhaka,Service,300000,NaN,10.0,2024-03-10
3,L004,Tanvir,Sylhet,Trading,750000,600000.0,95.0,2022-11-05
4,L005,Sadia,Khulna,Agriculture,200000,180000.0,NaN,2024-05-01
5,L006,Arif,Dhaka,Service,950000,900000.0,200.0,2021-08-18
6,L007,Mitu,Sylhet,Trading,400000,150000.0,0.0,2024-02-28
7,L008,Jamal,Chattogram,Manufacturing,1500000,NaN,30.0,2023-01-12
8,L009,Farzana,Khulna,Agriculture,250000,240000.0,400.0,2020-09-30
9,L005,Sadia,Khulna,Agriculture,200000,180000.0,NaN,2024-05-01


### 1.2 First look at the data

| Method | What it tells you |
|---|---|
| `df.head(n)` / `df.tail(n)` | First / last n rows |
| `df.shape` | (rows, columns) |
| `df.info()` | Column names, data types, non-null counts |
| `df.describe()` | Summary statistics for numeric columns |
| `df.columns` | List of column names |

In [3]:
print("Shape:", df.shape)
df.info()

Shape: (10, 8)
<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   loan_id        10 non-null     str    
 1   customer       10 non-null     str    
 2   branch         10 non-null     str    
 3   sector         10 non-null     str    
 4   loan_amount    10 non-null     str    
 5   outstanding    8 non-null      float64
 6   overdue_days   8 non-null      float64
 7   disburse_date  10 non-null     str    
dtypes: float64(2), str(6)
memory usage: 772.0 bytes


In [4]:
df.describe()

,outstanding,overdue_days
count,8.000000e+00,8.00000
mean,4.712500e+05,97.50000
std,3.634532e+05,139.38641
min,1.500000e+05,0.00000
25%,1.800000e+05,7.50000
50%,3.300000e+05,37.50000
75%,6.750000e+05,121.25000
max,1.100000e+06,400.00000


## 2. Selecting Columns

| Syntax | Returns |
|---|---|
| `df["col"]` | a **Series** (one column) |
| `df[["col1", "col2"]]` | a **DataFrame** (note the double brackets) |
| `df.loc[:, "col1":"col3"]` | a slice of columns by name |
| `df.iloc[:, 0:3]` | a slice of columns by position |
| `df.select_dtypes(include="number")` | columns by data type |

In [5]:
# Single column -> Series
df["customer"]

0      Rahim
1      Karim
2     Nusrat
3     Tanvir
4      Sadia
5       Arif
6       Mitu
7      Jamal
8    Farzana
9      Sadia
Name: customer, dtype: str

In [6]:
# Multiple columns -> DataFrame
df[["customer", "branch", "outstanding"]]

,customer,branch,outstanding
0,Rahim,Dhaka,420000.0
1,Karim,Chattogram,1100000.0
2,Nusrat,Dhaka,NaN
3,Tanvir,Sylhet,600000.0
4,Sadia,Khulna,180000.0
5,Arif,Dhaka,900000.0
6,Mitu,Sylhet,150000.0
7,Jamal,Chattogram,NaN
8,Farzana,Khulna,240000.0
9,Sadia,Khulna,180000.0


In [7]:
# Label-based slice (inclusive of both ends)
df.loc[:, "customer":"sector"].head(3)

,customer,branch,sector
0,Rahim,Dhaka,Trading
1,Karim,Chattogram,Manufacturing
2,Nusrat,Dhaka,Service


In [8]:
# Position-based slice (end is exclusive)
df.iloc[:, 0:3].head(3)

,loan_id,customer,branch
0,L001,Rahim,Dhaka
1,L002,Karim,Chattogram
2,L003,Nusrat,Dhaka


In [9]:
# Select only numeric columns
df.select_dtypes(include="number").head(3)

,outstanding,overdue_days
0,420000.0,0.0
1,1100000.0,45.0
2,NaN,10.0


## 3. Filtering Rows

Filtering uses a **boolean mask**: a True/False Series the same length as the DataFrame. Rows where the mask is `True` are kept.

- Combine conditions with `&` (and), `|` (or), `~` (not)
- **Always wrap each condition in parentheses**
- Useful helpers: `.isin()`, `.between()`, `.str.contains()`, `.isna()`, `.query()`

In [10]:
# The mask itself
mask = df["branch"] == "Dhaka"
mask

0     True
1    False
2     True
3    False
4    False
5     True
6    False
7    False
8    False
9    False
Name: branch, dtype: bool

In [11]:
# Apply the mask
df[mask]

,loan_id,customer,branch,sector,loan_amount,outstanding,overdue_days,disburse_date
0,L001,Rahim,Dhaka,Trading,500000,420000.0,0.0,2024-01-15
2,L003,Nusrat,Dhaka,Service,300000,NaN,10.0,2024-03-10
5,L006,Arif,Dhaka,Service,950000,900000.0,200.0,2021-08-18


In [12]:
# Multiple conditions: Dhaka OR Sylhet branch AND outstanding above 3 lakh
df[(df["branch"].isin(["Dhaka", "Sylhet"])) & (df["outstanding"] > 300000)]

,loan_id,customer,branch,sector,loan_amount,outstanding,overdue_days,disburse_date
0,L001,Rahim,Dhaka,Trading,500000,420000.0,0.0,2024-01-15
3,L004,Tanvir,Sylhet,Trading,750000,600000.0,95.0,2022-11-05
5,L006,Arif,Dhaka,Service,950000,900000.0,200.0,2021-08-18


In [13]:
# Range filter
df[df["overdue_days"].between(30, 100)]

,loan_id,customer,branch,sector,loan_amount,outstanding,overdue_days,disburse_date
1,L002,Karim,Chattogram,Manufacturing,1200000,1100000.0,45.0,2023-06-20
3,L004,Tanvir,Sylhet,Trading,750000,600000.0,95.0,2022-11-05
7,L008,Jamal,Chattogram,Manufacturing,1500000,NaN,30.0,2023-01-12


In [14]:
# NOT operator: everything except Trading
df[~(df["sector"] == "Trading")]

,loan_id,customer,branch,sector,loan_amount,outstanding,overdue_days,disburse_date
1,L002,Karim,Chattogram,Manufacturing,1200000,1100000.0,45.0,2023-06-20
2,L003,Nusrat,Dhaka,Service,300000,NaN,10.0,2024-03-10
4,L005,Sadia,Khulna,Agriculture,200000,180000.0,NaN,2024-05-01
5,L006,Arif,Dhaka,Service,950000,900000.0,200.0,2021-08-18
7,L008,Jamal,Chattogram,Manufacturing,1500000,NaN,30.0,2023-01-12
8,L009,Farzana,Khulna,Agriculture,250000,240000.0,400.0,2020-09-30
9,L005,Sadia,Khulna,Agriculture,200000,180000.0,NaN,2024-05-01


In [15]:
# String matching
df[df["customer"].str.contains("a", case=False)]

,loan_id,customer,branch,sector,loan_amount,outstanding,overdue_days,disburse_date
0,L001,Rahim,Dhaka,Trading,500000,420000.0,0.0,2024-01-15
1,L002,Karim,Chattogram,Manufacturing,1200000,1100000.0,45.0,2023-06-20
2,L003,Nusrat,Dhaka,Service,300000,NaN,10.0,2024-03-10
3,L004,Tanvir,Sylhet,Trading,750000,600000.0,95.0,2022-11-05
4,L005,Sadia,Khulna,Agriculture,200000,180000.0,NaN,2024-05-01
5,L006,Arif,Dhaka,Service,950000,900000.0,200.0,2021-08-18
7,L008,Jamal,Chattogram,Manufacturing,1500000,NaN,30.0,2023-01-12
8,L009,Farzana,Khulna,Agriculture,250000,240000.0,400.0,2020-09-30
9,L005,Sadia,Khulna,Agriculture,200000,180000.0,NaN,2024-05-01


In [16]:
# .query() — SQL-like, often more readable
df.query("overdue_days > 90 and branch != 'Dhaka'")

,loan_id,customer,branch,sector,loan_amount,outstanding,overdue_days,disburse_date
3,L004,Tanvir,Sylhet,Trading,750000,600000.0,95.0,2022-11-05
8,L009,Farzana,Khulna,Agriculture,250000,240000.0,400.0,2020-09-30


In [17]:
# Filter + select columns in one step with .loc[rows, columns]
df.loc[df["overdue_days"] > 90, ["loan_id", "customer", "overdue_days"]]

,loan_id,customer,overdue_days
3,L004,Tanvir,95.0
5,L006,Arif,200.0
8,L009,Farzana,400.0


## 4. Changing Data Types

Wrong data types are one of the most common data problems. In our data:
- `loan_amount` is stored as **text** (`object`/`str`) — we cannot do maths on it
- `disburse_date` is **text** — we cannot compute ages or extract months
- `branch` / `sector` repeat a few values — `category` saves memory

| Method | Use |
|---|---|
| `.astype(type)` | Direct conversion (fails on bad values) |
| `pd.to_numeric(col, errors="coerce")` | Safe numeric conversion — bad values become `NaN` |
| `pd.to_datetime(col)` | Text → datetime |
| `.astype("category")` | Repeating labels → category |

In [18]:
print("Before:")
print(df.dtypes)

Before:
loan_id              str
customer             str
branch               str
sector               str
loan_amount          str
outstanding      float64
overdue_days     float64
disburse_date        str
dtype: object


In [19]:
df["loan_amount"]   = df["loan_amount"].astype(int)          # text -> integer
df["disburse_date"] = pd.to_datetime(df["disburse_date"])    # text -> datetime
df["branch"]        = df["branch"].astype("category")        # text -> category
df["sector"]        = df["sector"].astype("category")

print("After:")
print(df.dtypes)

After:
loan_id                     str
customer                    str
branch                 category
sector                 category
loan_amount               int64
outstanding             float64
overdue_days            float64
disburse_date    datetime64[us]
dtype: object


In [20]:
# Safe conversion example: errors="coerce" turns invalid entries into NaN instead of crashing
messy = pd.Series(["100", "250", "abc", "400"])
pd.to_numeric(messy, errors="coerce")

0    100.0
1    250.0
2      NaN
3    400.0
dtype: float64

In [21]:
# Now date operations work
df["disburse_date"].dt.year.head()

0    2024
1    2023
2    2024
3    2022
4    2024
Name: disburse_date, dtype: int32

## 5. Creating New Columns

A new column is created simply by assigning to a column name that doesn't exist yet.

- **Arithmetic** between columns (vectorised — no loops needed)
- **`.apply()`** with a function for custom logic (slower; use when vectorised isn't possible)
- **`.assign()`** to create several columns in a chain, returning a new DataFrame
- **`pd.cut()`** to bin a number into ranges

In [22]:
# Arithmetic
df["repaid"] = df["loan_amount"] - df["outstanding"]
df["repaid_pct"] = (df["repaid"] / df["loan_amount"] * 100).round(1)

# From a date column
df["disburse_year"] = df["disburse_date"].dt.year

df[["loan_id", "loan_amount", "outstanding", "repaid", "repaid_pct", "disburse_year"]]

,loan_id,loan_amount,outstanding,repaid,repaid_pct,disburse_year
0,L001,500000,420000.0,80000.0,16.0,2024
1,L002,1200000,1100000.0,100000.0,8.3,2023
2,L003,300000,NaN,NaN,NaN,2024
3,L004,750000,600000.0,150000.0,20.0,2022
4,L005,200000,180000.0,20000.0,10.0,2024
5,L006,950000,900000.0,50000.0,5.3,2021
6,L007,400000,150000.0,250000.0,62.5,2024
7,L008,1500000,NaN,NaN,NaN,2023
8,L009,250000,240000.0,10000.0,4.0,2020
9,L005,200000,180000.0,20000.0,10.0,2024


In [23]:
# .apply() with a custom function
def size_label(amount):
    if amount >= 1_000_000:
        return "Large"
    elif amount >= 500_000:
        return "Medium"
    return "Small"

df["loan_size"] = df["loan_amount"].apply(size_label)
df[["loan_id", "loan_amount", "loan_size"]]

,loan_id,loan_amount,loan_size
0,L001,500000,Medium
1,L002,1200000,Large
2,L003,300000,Small
3,L004,750000,Medium
4,L005,200000,Small
5,L006,950000,Medium
6,L007,400000,Small
7,L008,1500000,Large
8,L009,250000,Small
9,L005,200000,Small


In [24]:
# pd.cut() — binning into ranges
df["overdue_bucket"] = pd.cut(
    df["overdue_days"],
    bins=[-1, 0, 30, 90, 180, np.inf],
    labels=["Current", "1-30", "31-90", "91-180", "180+"],
)
df[["loan_id", "overdue_days", "overdue_bucket"]]

,loan_id,overdue_days,overdue_bucket
0,L001,0.0,Current
1,L002,45.0,31-90
2,L003,10.0,1-30
3,L004,95.0,91-180
4,L005,NaN,NaN
5,L006,200.0,180+
6,L007,0.0,Current
7,L008,30.0,1-30
8,L009,400.0,180+
9,L005,NaN,NaN


In [25]:
# .assign() — chainable, returns a NEW DataFrame (original unchanged)
df.assign(
    amount_lakh = df["loan_amount"] / 100_000,
    customer_upper = df["customer"].str.upper(),
)[["loan_id", "amount_lakh", "customer_upper"]].head()

,loan_id,amount_lakh,customer_upper
0,L001,5.0,RAHIM
1,L002,12.0,KARIM
2,L003,3.0,NUSRAT
3,L004,7.5,TANVIR
4,L005,2.0,SADIA


## 6. Assigning Values

To **change** existing values, use indexers — not chained indexing.

| Indexer | Selects by | Example |
|---|---|---|
| `.loc[row_label, col_label]` | labels / boolean masks | `df.loc[df["x"] > 5, "y"] = 0` |
| `.iloc[row_pos, col_pos]` | integer positions | `df.iloc[0, 2] = "A"` |
| `.at[row_label, col]` | a single cell (fast) | `df.at[3, "y"] = 10` |

⚠️ **Avoid chained assignment** like `df[df["x"] > 5]["y"] = 0` — it modifies a temporary copy and silently does nothing (Pandas 3 makes this always fail to update the original).

In [26]:
work = df.copy()   # work on a copy so the original stays intact

# Conditional assignment with .loc
work["status"] = "Regular"
work.loc[work["overdue_days"] > 90, "status"] = "Classified"
work[["loan_id", "overdue_days", "status"]]

,loan_id,overdue_days,status
0,L001,0.0,Regular
1,L002,45.0,Regular
2,L003,10.0,Regular
3,L004,95.0,Classified
4,L005,NaN,Regular
5,L006,200.0,Classified
6,L007,0.0,Regular
7,L008,30.0,Regular
8,L009,400.0,Classified
9,L005,NaN,Regular


In [27]:
# Single cell by label (.at) and by position (.iat / .iloc)
work.at[0, "customer"] = "Rahim Uddin"
work.iloc[1, work.columns.get_loc("customer")] = "Karim Ali"
work[["loan_id", "customer"]].head(3)

,loan_id,customer
0,L001,Rahim Uddin
1,L002,Karim Ali
2,L003,Nusrat


In [28]:
# Replace values using a mapping
work["branch_code"] = work["branch"].map({"Dhaka": "DHK", "Chattogram": "CTG", "Sylhet": "SYL", "Khulna": "KHL"})
work[["branch", "branch_code"]].head()

,branch,branch_code
0,Dhaka,DHK
1,Chattogram,CTG
2,Dhaka,DHK
3,Sylhet,SYL
4,Khulna,KHL


## 7. Conditional Selection with NumPy

NumPy functions are **vectorised** — much faster than `.apply()` on large data.

- **`np.where(condition, value_if_true, value_if_false)`** → two outcomes (like Excel `IF`)
- **`np.select(conditions, choices, default)`** → many outcomes (like nested `IF` / `CASE WHEN`). Conditions are checked in order; the first `True` wins.

In [29]:
# np.where — two outcomes
df["is_npl"] = np.where(df["overdue_days"] > 90, "NPL", "Performing")
df[["loan_id", "overdue_days", "is_npl"]]

,loan_id,overdue_days,is_npl
0,L001,0.0,Performing
1,L002,45.0,Performing
2,L003,10.0,Performing
3,L004,95.0,NPL
4,L005,NaN,Performing
5,L006,200.0,NPL
6,L007,0.0,Performing
7,L008,30.0,Performing
8,L009,400.0,NPL
9,L005,NaN,Performing


In [30]:
# np.where can be nested, but np.select is cleaner for 3+ outcomes
conditions = [
    df["overdue_days"] == 0,
    df["overdue_days"] <= 60,
    df["overdue_days"] <= 180,
    df["overdue_days"] > 180,
]
choices = ["Standard", "SMA", "Substandard", "Bad/Loss"]

df["classification"] = np.select(conditions, choices, default="Unknown")
df[["loan_id", "overdue_days", "classification"]]

,loan_id,overdue_days,classification
0,L001,0.0,Standard
1,L002,45.0,SMA
2,L003,10.0,SMA
3,L004,95.0,Substandard
4,L005,NaN,Unknown
5,L006,200.0,Bad/Loss
6,L007,0.0,Standard
7,L008,30.0,SMA
8,L009,400.0,Bad/Loss
9,L005,NaN,Unknown


In [31]:
# NumPy for numeric selection: clip, max, log
print("Max outstanding :", np.max(df["outstanding"]))
print("Mean (ignores NaN):", np.nanmean(df["outstanding"]).round(0))

df["overdue_capped"] = np.clip(df["overdue_days"], 0, 180)   # cap extreme values
df[["loan_id", "overdue_days", "overdue_capped"]].head()

Max outstanding : 1100000.0
Mean (ignores NaN): 471250.0


,loan_id,overdue_days,overdue_capped
0,L001,0.0,0.0
1,L002,45.0,45.0
2,L003,10.0,10.0
3,L004,95.0,95.0
4,L005,NaN,NaN


## 8. Sorting Values

- `df.sort_values(by="col")` — ascending by default
- `ascending=False` — descending
- Multiple columns: `by=["a", "b"], ascending=[True, False]`
- `na_position="first"/"last"` — where missing values go
- `df.nlargest(n, "col")` / `df.nsmallest(n, "col")` — quick top-N
- `df.sort_index()` — back to original row order

In [32]:
df.sort_values("outstanding", ascending=False)[["loan_id", "customer", "outstanding"]]

,loan_id,customer,outstanding
1,L002,Karim,1100000.0
5,L006,Arif,900000.0
3,L004,Tanvir,600000.0
0,L001,Rahim,420000.0
8,L009,Farzana,240000.0
4,L005,Sadia,180000.0
9,L005,Sadia,180000.0
6,L007,Mitu,150000.0
2,L003,Nusrat,NaN
7,L008,Jamal,NaN


In [33]:
# Sort by branch (A→Z), then outstanding (high→low) within each branch
df.sort_values(["branch", "outstanding"], ascending=[True, False])[["branch", "customer", "outstanding"]]

,branch,customer,outstanding
1,Chattogram,Karim,1100000.0
7,Chattogram,Jamal,NaN
5,Dhaka,Arif,900000.0
0,Dhaka,Rahim,420000.0
2,Dhaka,Nusrat,NaN
8,Khulna,Farzana,240000.0
4,Khulna,Sadia,180000.0
9,Khulna,Sadia,180000.0
3,Sylhet,Tanvir,600000.0
6,Sylhet,Mitu,150000.0


In [34]:
# Top 3 by overdue days
df.nlargest(3, "overdue_days")[["loan_id", "customer", "overdue_days"]]

,loan_id,customer,overdue_days
8,L009,Farzana,400.0
5,L006,Arif,200.0
3,L004,Tanvir,95.0


## 9. Removing Duplicates

- `df.duplicated()` → boolean mask marking repeated rows
- `df.drop_duplicates()` → removes rows where **all** columns match
- `subset=["col"]` → consider only certain columns
- `keep="first"` (default) / `"last"` / `False` (drop all copies)

Our data has `L005` (Sadia) entered twice.

In [35]:
print("Duplicate rows (all columns):", df.duplicated().sum())
df[df.duplicated(subset=["loan_id"], keep=False)][["loan_id", "customer", "branch"]]

Duplicate rows (all columns): 1


,loan_id,customer,branch
4,L005,Sadia,Khulna
9,L005,Sadia,Khulna


In [36]:
before = len(df)
df = df.drop_duplicates(subset=["loan_id"], keep="first").reset_index(drop=True)
print(f"Rows before: {before}  |  after: {len(df)}")

Rows before: 10  |  after: 9


## 10. Imputing Missing Values

**Step 1 — Find them:** `df.isna().sum()`

**Step 2 — Choose a strategy:**

| Strategy | Code | When to use |
|---|---|---|
| Drop rows | `df.dropna(subset=[...])` | Few missing rows, not critical |
| Constant | `fillna(0)` | Missing genuinely means zero |
| Mean | `fillna(col.mean())` | Numeric, no strong outliers |
| Median | `fillna(col.median())` | Numeric **with outliers** (safer) |
| Mode | `fillna(col.mode()[0])` | Categorical data |
| Group-wise | `groupby(...).transform("median")` | Values differ by group |
| Forward/back fill | `ffill()` / `bfill()` | Time-series |
| Interpolate | `interpolate()` | Ordered numeric data |

💡 Keep a flag column (`was_missing`) so the model/analyst knows which values were imputed.

In [37]:
df.isna().sum()

loan_id           0
customer          0
branch            0
sector            0
loan_amount       0
outstanding       2
overdue_days      1
disburse_date     0
repaid            2
repaid_pct        2
disburse_year     0
loan_size         0
overdue_bucket    1
is_npl            0
classification    0
overdue_capped    1
dtype: int64

In [38]:
# Flag before imputing
df["outstanding_was_missing"] = df["outstanding"].isna()

# Group-wise median: fill outstanding using the median of the same sector
df["outstanding"] = df["outstanding"].fillna(
    df.groupby("sector", observed=True)["outstanding"].transform("median")
)

# Assume a missing overdue_days means no overdue record -> 0
df["overdue_days"] = df["overdue_days"].fillna(0)

df[["loan_id", "sector", "outstanding", "outstanding_was_missing", "overdue_days"]]

,loan_id,sector,outstanding,outstanding_was_missing,overdue_days
0,L001,Trading,420000.0,False,0.0
1,L002,Manufacturing,1100000.0,False,45.0
2,L003,Service,900000.0,True,10.0
3,L004,Trading,600000.0,False,95.0
4,L005,Agriculture,180000.0,False,0.0
5,L006,Service,900000.0,False,200.0
6,L007,Trading,150000.0,False,0.0
7,L008,Manufacturing,1100000.0,True,30.0
8,L009,Agriculture,240000.0,False,400.0


In [39]:
# Other strategies, shown on a small Series
s = pd.Series([10, np.nan, 30, np.nan, 50])
pd.DataFrame({
    "original":    s,
    "mean":        s.fillna(s.mean()),
    "median":      s.fillna(s.median()),
    "ffill":       s.ffill(),
    "interpolate": s.interpolate(),
})

,original,mean,median,ffill,interpolate
0,10.0,10.0,10.0,10.0,10.0
1,NaN,30.0,30.0,10.0,20.0
2,30.0,30.0,30.0,30.0,30.0
3,NaN,30.0,30.0,30.0,40.0
4,50.0,50.0,50.0,50.0,50.0


In [40]:
# Recompute derived columns now that gaps are filled
df["repaid"] = df["loan_amount"] - df["outstanding"]
conditions = [
    df["overdue_days"] == 0, df["overdue_days"] <= 60,
    df["overdue_days"] <= 180, df["overdue_days"] > 180,
]
df["classification"] = np.select(conditions, choices, default="Unknown")

print("Missing values left:")
print(df[["outstanding", "overdue_days", "repaid"]].isna().sum())

Missing values left:
outstanding     0
overdue_days    0
repaid          0
dtype: int64


## 11. Pivoting

Pivoting reshapes data from **long** (one row per record) to **wide** (a summary grid) — like an Excel Pivot Table.

| Function | Aggregates? | Notes |
|---|---|---|
| `df.pivot(index, columns, values)` | ❌ No | Fails if index/column pairs repeat |
| `df.pivot_table(index, columns, values, aggfunc)` | ✅ Yes | Handles duplicates; supports totals (`margins=True`) |
| `pd.crosstab(a, b)` | ✅ Counts | Quick frequency table |
| `df.melt(...)` | — | The reverse: wide → long |

In [41]:
# pivot_table: total outstanding by branch x sector
pt = pd.pivot_table(
    df,
    index="branch",
    columns="sector",
    values="outstanding",
    aggfunc="sum",
    fill_value=0,
    margins=True,
    margins_name="Total",
    observed=True,
)
pt

sector,Agriculture,Manufacturing,Service,Trading,Total
branch,,,,,
Chattogram,0.0,2200000.0,0.0,0.0,2200000.0
Dhaka,0.0,0.0,1800000.0,420000.0,2220000.0
Khulna,420000.0,0.0,0.0,0.0,420000.0
Sylhet,0.0,0.0,0.0,750000.0,750000.0
Total,420000.0,2200000.0,1800000.0,1170000.0,5590000.0


In [42]:
# Multiple aggregations
pd.pivot_table(
    df, index="branch", values=["outstanding", "overdue_days"],
    aggfunc={"outstanding": "sum", "overdue_days": "mean"}, observed=True,
).round(1)

,outstanding,overdue_days
branch,,
Chattogram,2200000.0,37.5
Dhaka,2220000.0,70.0
Khulna,420000.0,200.0
Sylhet,750000.0,47.5


In [43]:
# crosstab: count of loans by branch and classification
pd.crosstab(df["branch"], df["classification"], margins=True)

classification,Bad/Loss,SMA,Standard,Substandard,All
branch,,,,,
Chattogram,0,2,0,0,2
Dhaka,1,1,1,0,3
Khulna,1,0,1,0,2
Sylhet,0,0,1,1,2
All,2,3,3,1,9


In [44]:
# pivot (no aggregation): works when each index/column pair is unique
monthly = pd.DataFrame({
    "month":  ["Jan", "Jan", "Feb", "Feb"],
    "branch": ["Dhaka", "Sylhet", "Dhaka", "Sylhet"],
    "npl":    [120, 80, 135, 76],
})
wide = monthly.pivot(index="month", columns="branch", values="npl")
wide

branch,Dhaka,Sylhet
month,,
Feb,135,76
Jan,120,80


In [45]:
# melt: wide -> long (the reverse of pivot)
wide.reset_index().melt(id_vars="month", var_name="branch", value_name="npl")

,month,branch,npl
0,Feb,Dhaka,135
1,Jan,Dhaka,120
2,Feb,Sylhet,76
3,Jan,Sylhet,80


## Final Cleaned Dataset

In [46]:
final = df[["loan_id", "customer", "branch", "sector", "loan_amount",
            "outstanding", "overdue_days", "classification", "loan_size"]] \
          .sort_values("overdue_days", ascending=False) \
          .reset_index(drop=True)
final

,loan_id,customer,branch,sector,loan_amount,outstanding,overdue_days,classification,loan_size
0,L009,Farzana,Khulna,Agriculture,250000,240000.0,400.0,Bad/Loss,Small
1,L006,Arif,Dhaka,Service,950000,900000.0,200.0,Bad/Loss,Medium
2,L004,Tanvir,Sylhet,Trading,750000,600000.0,95.0,Substandard,Medium
3,L002,Karim,Chattogram,Manufacturing,1200000,1100000.0,45.0,SMA,Large
4,L008,Jamal,Chattogram,Manufacturing,1500000,1100000.0,30.0,SMA,Large
5,L003,Nusrat,Dhaka,Service,300000,900000.0,10.0,SMA,Small
6,L001,Rahim,Dhaka,Trading,500000,420000.0,0.0,Standard,Medium
7,L005,Sadia,Khulna,Agriculture,200000,180000.0,0.0,Standard,Small
8,L007,Mitu,Sylhet,Trading,400000,150000.0,0.0,Standard,Small


## 12. More Essential Functions

This section covers six everyday functions for tidying and combining data:

| Function | One-line purpose |
|---|---|
| `rename()` | Change column or index labels |
| `drop()` | Remove columns or rows |
| `concat()` | Stack DataFrames on top of / beside each other |
| `join()` / `merge()` | Combine DataFrames on a matching key |
| `replace()` / `map()` | Recode values |
| `value_counts()` | Count how often each value appears |

We'll work on a copy of the cleaned data: `demo = final.copy()`.

In [47]:
demo = final.copy()
demo.head()

,loan_id,customer,branch,sector,loan_amount,outstanding,overdue_days,classification,loan_size
0,L009,Farzana,Khulna,Agriculture,250000,240000.0,400.0,Bad/Loss,Small
1,L006,Arif,Dhaka,Service,950000,900000.0,200.0,Bad/Loss,Medium
2,L004,Tanvir,Sylhet,Trading,750000,600000.0,95.0,Substandard,Medium
3,L002,Karim,Chattogram,Manufacturing,1200000,1100000.0,45.0,SMA,Large
4,L008,Jamal,Chattogram,Manufacturing,1500000,1100000.0,30.0,SMA,Large


### 12.1 `rename()` — Change Column / Index Names

- Pass a **dictionary** `{old: new}` to `columns=` (or `index=`)
- Only listed names change; all others stay the same
- Pass a **function** to apply the same change to every name (e.g. `str.upper`)
- Returns a new DataFrame — assign it back (`df = df.rename(...)`)

💡 For renaming *all* columns at once, you can also assign a list: `df.columns = [...]`

In [48]:
# Rename specific columns
demo = demo.rename(columns={
    "customer": "customer_name",
    "loan_amount": "sanctioned_amount",
})
demo.columns.tolist()

['loan_id',
 'customer_name',
 'branch',
 'sector',
 'sanctioned_amount',
 'outstanding',
 'overdue_days',
 'classification',
 'loan_size']

In [49]:
# Apply a function to every column name
demo.rename(columns=str.upper).columns.tolist()

['LOAN_ID',
 'CUSTOMER_NAME',
 'BRANCH',
 'SECTOR',
 'SANCTIONED_AMOUNT',
 'OUTSTANDING',
 'OVERDUE_DAYS',
 'CLASSIFICATION',
 'LOAN_SIZE']

In [50]:
# Common cleanup: lowercase, strip spaces, replace spaces with underscores
raw = pd.DataFrame(columns=[" Loan ID", "Customer Name ", "Overdue Days"])
raw.columns = raw.columns.str.strip().str.lower().str.replace(" ", "_")
raw.columns.tolist()

['loan_id', 'customer_name', 'overdue_days']

In [51]:
# Rename index labels
demo.rename(index={0: "first_row"}).head(2)

,loan_id,customer_name,branch,sector,sanctioned_amount,outstanding,overdue_days,classification,loan_size
first_row,L009,Farzana,Khulna,Agriculture,250000,240000.0,400.0,Bad/Loss,Small
1,L006,Arif,Dhaka,Service,950000,900000.0,200.0,Bad/Loss,Medium


### 12.2 `drop()` — Remove Columns or Rows

| Goal | Code |
|---|---|
| Drop columns | `df.drop(columns=["a", "b"])` |
| Drop rows by index label | `df.drop(index=[0, 3])` |
| Drop rows by condition | `df.drop(df[condition].index)` |
| Ignore missing names | `errors="ignore"` |

💡 To drop rows by condition, filtering is usually simpler: `df[~condition]`.

In [52]:
# Drop columns
demo.drop(columns=["loan_size", "sector"]).head(3)

,loan_id,customer_name,branch,sanctioned_amount,outstanding,overdue_days,classification
0,L009,Farzana,Khulna,250000,240000.0,400.0,Bad/Loss
1,L006,Arif,Dhaka,950000,900000.0,200.0,Bad/Loss
2,L004,Tanvir,Sylhet,750000,600000.0,95.0,Substandard


In [53]:
# Drop rows by index label
demo.drop(index=[0, 1]).head(3)

,loan_id,customer_name,branch,sector,sanctioned_amount,outstanding,overdue_days,classification,loan_size
2,L004,Tanvir,Sylhet,Trading,750000,600000.0,95.0,Substandard,Medium
3,L002,Karim,Chattogram,Manufacturing,1200000,1100000.0,45.0,SMA,Large
4,L008,Jamal,Chattogram,Manufacturing,1500000,1100000.0,30.0,SMA,Large


In [54]:
# Drop rows by condition: remove all Bad/Loss loans
cleaned = demo.drop(demo[demo["classification"] == "Bad/Loss"].index)
print("Rows before:", len(demo), "| after:", len(cleaned))

# Same result using a filter
same = demo[demo["classification"] != "Bad/Loss"]
print("Filter method rows:", len(same))

Rows before: 9 | after: 7
Filter method rows: 7


In [55]:
# errors="ignore" prevents a KeyError when a column doesn't exist
demo.drop(columns=["not_a_column"], errors="ignore").shape

(9, 9)

### 12.3 `concat()` — Stack DataFrames

`pd.concat()` glues DataFrames together **without matching on a key**.

| Parameter | Effect |
|---|---|
| `axis=0` (default) | Stack **vertically** — add rows |
| `axis=1` | Place **side by side** — add columns (aligned on index) |
| `ignore_index=True` | Renumber the index 0…n |
| `keys=[...]` | Label where each part came from |

Typical use: combining monthly files that share the same columns.

In [56]:
jan = pd.DataFrame({"loan_id": ["L001", "L002"], "outstanding": [420000, 1100000]})
feb = pd.DataFrame({"loan_id": ["L003", "L004"], "outstanding": [310000, 600000]})

# Vertical stack
pd.concat([jan, feb], ignore_index=True)

,loan_id,outstanding
0,L001,420000
1,L002,1100000
2,L003,310000
3,L004,600000


In [57]:
# keys= adds a label showing the source of each row
pd.concat([jan, feb], keys=["Jan", "Feb"])

loan_id  outstanding
Jan 0    L001       420000
    1    L002      1100000
Feb 0    L003       310000
    1    L004       600000

In [58]:
# Mismatched columns are filled with NaN
mar = pd.DataFrame({"loan_id": ["L005"], "overdue_days": [15]})
pd.concat([jan, mar], ignore_index=True)

,loan_id,outstanding,overdue_days
0,L001,420000.0,NaN
1,L002,1100000.0,NaN
2,L005,NaN,15.0


In [59]:
# Horizontal stack (axis=1) — rows are aligned by index position/label
scores = pd.DataFrame({"credit_score": [720, 650]})
pd.concat([jan, scores], axis=1)

,loan_id,outstanding,credit_score
0,L001,420000,720
1,L002,1100000,650


### 12.4 `join()` and `merge()` — Combine on a Key

Unlike `concat`, these **match rows** using a key — like SQL joins or Excel `VLOOKUP`.

- **`df.join(other)`** → matches on the **index** (quick when keys are already the index)
- **`pd.merge(left, right, on="key")`** → matches on **columns** (more flexible, most common)

| `how=` | Keeps |
|---|---|
| `"left"` | All rows from the left table (default for `join`) |
| `"right"` | All rows from the right table |
| `"inner"` | Only matching rows (default for `merge`) |
| `"outer"` | All rows from both |

In [60]:
loans = pd.DataFrame({
    "loan_id": ["L001", "L002", "L003", "L010"],
    "outstanding": [420000, 1100000, 310000, 90000],
})
officers = pd.DataFrame({
    "loan_id": ["L001", "L002", "L003", "L004"],
    "officer": ["Hasan", "Rupa", "Hasan", "Nabil"],
})

# join() works on the index, so set the key as index first
loans.set_index("loan_id").join(officers.set_index("loan_id"), how="left")

,outstanding,officer
loan_id,,
L001,420000,Hasan
L002,1100000,Rupa
L003,310000,Hasan
L010,90000,NaN


In [61]:
# merge() works directly on columns — compare the different 'how' options
for how in ["left", "inner", "outer"]:
    print(f"--- how='{how}' ---")
    print(pd.merge(loans, officers, on="loan_id", how=how), "\n")

--- how='left' ---
  loan_id  outstanding officer
0    L001       420000   Hasan
1    L002      1100000    Rupa
2    L003       310000   Hasan
3    L010        90000     NaN 

--- how='inner' ---
  loan_id  outstanding officer
0    L001       420000   Hasan
1    L002      1100000    Rupa
2    L003       310000   Hasan 

--- how='outer' ---
  loan_id  outstanding officer
0    L001     420000.0   Hasan
1    L002    1100000.0    Rupa
2    L003     310000.0   Hasan
3    L004          NaN   Nabil
4    L010      90000.0     NaN 



In [62]:
# When key columns have different names, use left_on / right_on
staff = officers.rename(columns={"loan_id": "account_no"})
pd.merge(loans, staff, left_on="loan_id", right_on="account_no", how="left")

,loan_id,outstanding,account_no,officer
0,L001,420000,L001,Hasan
1,L002,1100000,L002,Rupa
2,L003,310000,L003,Hasan
3,L010,90000,NaN,NaN


In [63]:
# indicator=True shows where each row came from — great for reconciliation
pd.merge(loans, officers, on="loan_id", how="outer", indicator=True)

,loan_id,outstanding,officer,_merge
0,L001,420000.0,Hasan,both
1,L002,1100000.0,Rupa,both
2,L003,310000.0,Hasan,both
3,L004,NaN,Nabil,right_only
4,L010,90000.0,NaN,left_only


### 12.5 `replace()` vs `map()` — Recode Values

Both change values using a mapping, but they handle **unmatched values** differently:

| | `replace()` | `map()` |
|---|---|---|
| Works on | Series **or** DataFrame | Series only |
| Unmatched values | **Kept as they are** | Become **NaN** |
| Accepts a function | No | Yes |
| Best for | Fixing a few specific values | Translating every value (lookup) |

In [64]:
s = pd.Series(["Dhaka", "Ctg", "Sylhet", "Ctg"])

pd.DataFrame({
    "original": s,
    "replace":  s.replace({"Ctg": "Chattogram"}),   # unmatched kept
    "map":      s.map({"Ctg": "Chattogram"}),       # unmatched -> NaN
})

,original,replace,map
0,Dhaka,Dhaka,NaN
1,Ctg,Chattogram,Chattogram
2,Sylhet,Sylhet,NaN
3,Ctg,Chattogram,Chattogram


In [65]:
# replace() on a whole DataFrame, e.g. cleaning placeholder values
messy = pd.DataFrame({"amount": [100, -999, 250], "status": ["OK", "N/A", "OK"]})
messy.replace({-999: np.nan, "N/A": np.nan})

,amount,status
0,100.0,OK
1,NaN,NaN
2,250.0,OK


In [66]:
# map() as a lookup: classification -> provision rate
provision_rate = {"Standard": 0.01, "SMA": 0.05, "Substandard": 0.20, "Bad/Loss": 1.00}
demo["provision_rate"] = demo["classification"].map(provision_rate)
demo["provision"] = demo["outstanding"] * demo["provision_rate"]
demo[["loan_id", "classification", "outstanding", "provision_rate", "provision"]]

,loan_id,classification,outstanding,provision_rate,provision
0,L009,Bad/Loss,240000.0,1.00,240000.0
1,L006,Bad/Loss,900000.0,1.00,900000.0
2,L004,Substandard,600000.0,0.20,120000.0
3,L002,SMA,1100000.0,0.05,55000.0
4,L008,SMA,1100000.0,0.05,55000.0
5,L003,SMA,900000.0,0.05,45000.0
6,L001,Standard,420000.0,0.01,4200.0
7,L005,Standard,180000.0,0.01,1800.0
8,L007,Standard,150000.0,0.01,1500.0


In [67]:
# map() with a function
demo["customer_name"].map(lambda name: name[:3].upper()).head()

0    FAR
1    ARI
2    TAN
3    KAR
4    JAM
Name: customer_name, dtype: str

### 12.6 `value_counts()` — Frequency Counts

The fastest way to understand a categorical column during EDA.

| Parameter | Effect |
|---|---|
| `normalize=True` | Proportions instead of counts |
| `dropna=False` | Include missing values in the count |
| `ascending=True` | Least frequent first |
| `bins=n` | Group a numeric column into n ranges |

Also works on a DataFrame to count **combinations** of columns.

In [68]:
demo["branch"].value_counts()

branch
Dhaka         3
Chattogram    2
Khulna        2
Sylhet        2
Name: count, dtype: int64

In [69]:
# Percentages
(demo["classification"].value_counts(normalize=True) * 100).round(1)

classification
SMA            33.3
Standard       33.3
Bad/Loss       22.2
Substandard    11.1
Name: proportion, dtype: float64

In [70]:
# Include missing values
pd.Series(["A", "B", np.nan, "A", np.nan]).value_counts(dropna=False)

A      2
NaN    2
B      1
Name: count, dtype: int64

In [71]:
# Numeric column in bins
demo["outstanding"].value_counts(bins=3)

(149049.999, 466666.667]    4
(783333.333, 1100000.0]     4
(466666.667, 783333.333]    1
Name: count, dtype: int64

In [72]:
# Count combinations of two columns
demo[["branch", "classification"]].value_counts()

branch      classification
Chattogram  SMA               2
Khulna      Bad/Loss          1
Dhaka       Bad/Loss          1
Sylhet      Substandard       1
Dhaka       SMA               1
            Standard          1
Khulna      Standard          1
Sylhet      Standard          1
            Bad/Loss          0
Chattogram  Bad/Loss          0
Khulna      Substandard       0
Dhaka       Substandard       0
Chattogram  Substandard       0
Khulna      SMA               0
Sylhet      SMA               0
Chattogram  Standard          0
Name: count, dtype: int64

## 13. Practice Exercises

1. Select only `customer`, `sector`, and `repaid_pct` for loans disbursed in 2024.
2. Convert `overdue_days` to integer type.
3. Create a column `risk_flag` = `"High"` if classification is Substandard or Bad/Loss, else `"Low"` (use `np.where` with `.isin()`).
4. Set `loan_size` to `"Priority"` for all Agriculture loans using `.loc`.
5. Find the branch with the highest average outstanding (`groupby` + `sort_values`).
6. Build a pivot table: count of loans per `sector` × `loan_size`.
7. Introduce a missing `branch` value and impute it with the mode.
8. Rename `overdue_days` to `dpd` and drop the `loan_size` column.
9. Stack `jan` and `feb` with `concat`, then `merge` the result with `officers` using a left join.
10. Use `value_counts(normalize=True)` to find the share of loans in each sector.

## Quick Reference

```python
df["col"] / df[["a","b"]]                 # select
df[(cond1) & (cond2)]                     # filter
df["col"].astype(int); pd.to_datetime()   # dtype
df["new"] = df["a"] * df["b"]             # new column
df.loc[mask, "col"] = value               # assign
np.where(cond, a, b); np.select(c, ch)    # numpy selection
df.pivot_table(index, columns, values, aggfunc)
df.sort_values(by, ascending)
df.drop_duplicates(subset, keep)
df["col"].fillna(df["col"].median())      # impute
df.rename(columns={"old": "new"})         # rename
df.drop(columns=["a"])                    # drop
pd.concat([df1, df2], ignore_index=True)  # stack
pd.merge(a, b, on="key", how="left")      # join on key
s.replace({"x": "y"}); s.map(lookup)      # recode
df["col"].value_counts(normalize=True)    # frequency
```